In [1]:
!pip uninstall torch torchvision torchaudio pandas numpy -y

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install FlagEmbedding
!pip install pymilvus[milvus-lite]
!pip install milvus-lite

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: pandas 2.3.3
Uninstalling pandas-2.3.3:
  Successfully uninstalled pandas-2.3.3
Found existing installation: numpy 2.4.6
Uninstalling numpy-2.4.6:
  Successfully uninstalled numpy-2.4.6
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.8/289.8 MB 98.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 84.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 212.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 73.1 MB/s 

In [2]:
import json
import os
from pathlib import Path
from FlagEmbedding import BGEM3FlagModel

import shutil
from pathlib import Path
from pymilvus import DataType, MilvusClient
from pymilvus import DataType, Function, FunctionType, MilvusClient
from pymilvus import AnnSearchRequest, utility


DB_PATH = "demo_1.db"
DENSE_DIM = 1024

MODEL_NAME  = "BAAI/bge-m3"   # downloads ~2.2GB on first run, cached after
BATCH_SIZE  = 8                # keep low for CPU / Kaggle free tier
MAX_LENGTH  = 512              # 10-Q chunks are rarely longer than this
USE_FP16    = True            # True only if you have a GPU

In [3]:
_model = None

def get_model() -> BGEM3FlagModel:
    """
    Return the BGE-M3 model, loading it only on the first call.
    Subsequent calls return the cached instance.
    """
    global _model
    if _model is None:
        print(f"Loading BGE-M3 model: {MODEL_NAME} ...")
        _model = BGEM3FlagModel(
            MODEL_NAME,
            use_fp16=USE_FP16,
            device="cuda"
        )
        print("Model loaded.\n")
    return _model


# ── SPARSE VECTOR CONVERTER ───────────────────────────────────────────────────

def lexical_weights_to_sparse(lexical_weights: dict) -> dict:
    """
    Convert BGE-M3 lexical_weights output to Milvus Lite sparse format.

    BGE-M3 returns:  { "token_id_str": weight_float, ... }
    Milvus needs:    { "indices": [int, ...], "values": [float, ...] }
    """
    indices = [int(k)   for k in lexical_weights.keys()]
    values  = [float(v) for v in lexical_weights.values()]
    return {"indices": indices, "values": values}


# ── CORPUS EMBEDDER ───────────────────────────────────────────────────────────

def embed_corpus(texts: list, is_table: bool = False) -> list[dict]:
    """
    Embed a list of document chunks (corpus side).

    Uses encode_corpus() — BGE-M3 treats documents differently from queries
    internally (no instruction prefix added for documents).

    Returns list of:
      {
        "dense":  [float, ...]        # 1024-dim vector
        "sparse": {"indices": [...],  # token ids
                   "values":  [...]}  # term weights
      }
    """
    if is_table:
        # Serialize each table to a JSON string so the full structure is
        # embedded as one unit without splitting rows/columns.
        texts = [
            json.dumps(t, ensure_ascii=False) if isinstance(t, (dict, list)) else str(t)
            for t in texts
        ]

    model = get_model()

    print(f"  Embedding {len(texts)} {'table' if is_table else 'text'} chunks "
          f"(batch_size={BATCH_SIZE}) ...")

    output = model.encode_corpus(
        texts,
        batch_size          = BATCH_SIZE,
        max_length          = MAX_LENGTH,
        return_dense        = True,
        return_sparse       = True,
        return_colbert_vecs = False,   # not needed for POC
    )

    dense_vecs      = output["dense_vecs"]       # np.ndarray (N, 1024)
    lexical_weights = output["lexical_weights"]  # list of dicts

    results = []
    for i in range(len(texts)):
        results.append({
            "dense":  dense_vecs[i].tolist(),
            "sparse": lexical_weights_to_sparse(lexical_weights[i]),
        })

    print(f"  Done. Dense dim={len(results[0]['dense'])}  "
          f"Sparse tokens={len(results[0]['sparse']['indices'])}\n")

    return results


# ── QUERY EMBEDDER ────────────────────────────────────────────────────────────

def embed_query(query: str) -> dict:
    """
    Embed a single user query (query side).

    Uses encode_queries() — BGE-M3 adds an internal instruction prefix
    for queries to improve retrieval quality.

    Returns:
      {
        "dense":  [float, ...]        # 1024-dim vector
        "sparse": {"indices": [...],
                   "values":  [...]}
      }
    """
    model = get_model()

    output = model.encode_queries(
        [query],                       # always pass as list
        batch_size          = 1,
        max_length          = 512,
        return_dense        = True,
        return_sparse       = True,
        return_colbert_vecs = False,
    )

    dense_vecs      = output["dense_vecs"]
    lexical_weights = output["lexical_weights"]

    return {
        "dense":  dense_vecs[0].tolist(),
        "sparse": lexical_weights_to_sparse(lexical_weights[0]),
    }


def create_embedding():
    output_dir = "/kaggle/input/datasets/koushikcon/private-doc-rag-test/output"
    text_documents = []
    table_documents = []

    for filename in os.listdir(output_dir):
        if not filename.endswith(".json"):
            continue

        filepath = os.path.join(output_dir, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        metadata = data.get("metadata", {})
        texts: list[str] = []
        tables: list = []

        for element in data.get("elements", []):
            if element.get("type") == "text":
                texts.append(element.get("content", ""))
            elif element.get("type") == "table":
                tables.append(element.get("content", {}))

        text_corpus_embeddings = embed_corpus(texts) if texts else []
        table_corpus_embeddings = embed_corpus(tables, is_table=True) if tables else []

        for content, emb in zip(texts, text_corpus_embeddings):
            text_documents.append({
                "content": content,
                "embedding": emb,
                "year": metadata.get("year"),
                "quarter": metadata.get("quarter"),
                "ticker": metadata.get("ticker"),
            })

        for content, emb in zip(tables, table_corpus_embeddings):
            table_documents.append({
                "content": content,
                "embedding": emb,
                "year": metadata.get("year"),
                "quarter": metadata.get("quarter"),
                "ticker": metadata.get("ticker"),
            })
    return text_documents, table_documents


In [4]:

def _build_schema():
    schema = MilvusClient.create_schema(auto_id=True)
    schema.add_field("id", DataType.INT64, is_primary=True)
    schema.add_field("content", DataType.VARCHAR, max_length=65535, enable_analyzer=True, enable_match=True)
    schema.add_field("dense", DataType.FLOAT_VECTOR, dim=DENSE_DIM)
    schema.add_field("sparse", DataType.SPARSE_FLOAT_VECTOR)
    schema.add_field("metadata", DataType.JSON)
    return schema


def _build_index_params(client):
    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name="dense",
        index_type="IVF_FLAT",
        metric_type="COSINE",
        params={"nlist": 128},
    )
    index_params.add_index(
        field_name="sparse",
        index_type="SPARSE_INVERTED_INDEX",
        metric_type="IP",
        params={},
    )
    return index_params


def _sparse_to_dict(sparse: dict) -> dict:
    return {int(idx): float(val) for idx, val in zip(sparse["indices"], sparse["values"])}


def setup_collections(name: str, client: MilvusClient) -> MilvusClient:
    schema = MilvusClient.create_schema(auto_id=True)
    schema.add_field("id", DataType.INT64, is_primary=True)
    schema.add_field("year", DataType.INT64)
    schema.add_field("quarter", DataType.VARCHAR, max_length=10)
    schema.add_field("ticker", DataType.VARCHAR, max_length=20)
    schema.add_field("content", DataType.VARCHAR, max_length=65535, enable_analyzer=True, enable_match=True)
    schema.add_field("dense", DataType.FLOAT_VECTOR, dim=DENSE_DIM)
    schema.add_field("sparse", DataType.SPARSE_FLOAT_VECTOR)
    schema.add_function(Function(
        name="bm25",
        function_type=FunctionType.BM25,
        input_field_names=["content"],
        output_field_names=["sparse"],
    ))

    client.create_collection(name, schema=schema)
    sparse_params = client.prepare_index_params()
    sparse_params.add_index(
        field_name="sparse",
        index_name=f"{name}_sparse_index",
        index_type="SPARSE_INVERTED_INDEX",
        metric_type="BM25",
        params={"inverted_index_algo": "DAAT_MAXSCORE"},
    )
    client.create_index(name, sparse_params)
    dense_params = client.prepare_index_params()
    dense_params.add_index(
        field_name="dense",
        index_name=f"{name}_dense_index",
        index_type="AUTOINDEX",
        metric_type="IP",
    )
    client.create_index(name, dense_params)
    return client

def insert_documents(client: MilvusClient, text_documents: list, table_documents: list):
    text_rows = [
        {
            "content": doc["content"],
            "dense": doc["embedding"]["dense"],
            "year": int(doc["year"]),
            "quarter": doc["quarter"],
            "ticker": doc["ticker"],
        }
        for doc in text_documents
    ]

    table_rows = [
        {
            "content": json.dumps(doc["content"], ensure_ascii=False) if isinstance(doc["content"], (dict, list)) else str(doc["content"]),
            "dense": doc["embedding"]["dense"],
            "year": int(doc["year"]),
            "quarter": doc["quarter"],
            "ticker": doc["ticker"],
        }
        for doc in table_documents
    ]

    if text_rows:
        res = client.insert("text_chunks", text_rows)
        print(f"Inserted {len(text_rows)} text rows into text_chunks")
        client.flush(collection_name="text_chunks", timeout=600.0)

    if table_rows:
        res = client.insert("table_chunks", table_rows)
        print(f"Inserted {len(table_rows)} table rows into table_chunks")
        client.flush(collection_name="table_chunks", timeout=600.0)

In [5]:
db = Path(DB_PATH)
client = MilvusClient(DB_PATH)
setup_collections("text_chunks", client)
setup_collections("table_chunks", client)
print("list of all collections ")
print(client.list_collections())
text_documents, table_documents = create_embedding()
insert_documents(client, text_documents, table_documents)

# perform_search(client, "What was Apple's revenue in Q1 2023?", "text_chunks", top_k=5)

list of all collections 
['table_chunks', 'text_chunks']
Loading BGE-M3 model: BAAI/bge-m3 ...


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

  Done. Dense dim=1024  Sparse tokens=131

  Embedding 212 text chunks (batch_size=8) ...



pre tokenize: 100%|██████████| 27/27 [00:00<00:00, 519.80it/s]

Inference Embeddings: 100%|██████████| 27/27 [00:03<00:00,  8.51it/s]


  Done. Dense dim=1024  Sparse tokens=49

  Embedding 43 table chunks (batch_size=8) ...



pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 319.68it/s]

Inference Embeddings: 100%|██████████| 6/6 [00:01<00:00,  3.32it/s]
I0617 05:08:25.527409     121 chttp2_transport.cc:1369] ipv4:127.0.0.1:37399: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11}
E0617 05:08:25.527491     121 chttp2_transport.cc:1401] ipv4:127.0.0.1:37399: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


  Done. Dense dim=1024  Sparse tokens=105

Inserted 4082 text rows into text_chunks
Inserted 480 table rows into table_chunks


In [6]:
def perform_search(client: MilvusClient, query_text: str, collection_name: str, top_k: int = 5):
    client.load_collection(collection_name)
    query_embedding = embed_query(query_text)
    query_dense_vector = query_embedding["dense"]
    query_sparse_vector = _sparse_to_dict(query_embedding["sparse"])

    # semantic search (dense)
    search_param_1 = {
        "data": [query_dense_vector],
        "anns_field": "dense",
        "param": {"nprobe": 10},
        "limit": top_k,
    }
    request_1 = AnnSearchRequest(**search_param_1)

    # lexical search (sparse BGE-M3)
    search_param_2 = {
        "data": [query_text],
        "anns_field": "sparse",
        "param": {},
        "limit": top_k,
    }
    request_2 = AnnSearchRequest(**search_param_2)

    ranker = Function(
        name="rrf",
        input_field_names=[], # Must be an empty list
        function_type=FunctionType.RERANK,
        params={
            "reranker": "rrf", 
            "k": 100  # Optional
        }
    )

    res = client.hybrid_search(
        collection_name=collection_name,
        reqs=[request_1, request_2],
        ranker=ranker,
        limit=top_k
    )
    # print(res)
    for hits in res:
        print("TopK results:")
        print(hits)
        for hit in hits:
            print(hit)

    return hits

In [7]:
client.load_collection("text_chunks")
perform_search(client, "What was Apple's revenue in Q1 2023?", "table_chunks", top_k=5)

res = client.query(
    collection_name="text_chunks",
    filter="id >= 3", 
    output_fields=["id", "content"]
)
client.release_collection("text_chunks")


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 2165.36it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 65.95it/s]


TopK results:
[{'id': 125, 'distance': 0.019801979884505272, 'entity': {'id': 125, 'year': 2023, 'quarter': 'Q1', 'ticker': 'INTC', 'content': '| $18.4\n$11.7\nQ1 2022 Q1 2023 |  | 53.1%\n38.4%\n50.4%\n34.2%\nQ1 2022 Q1 2023 |  | $5.6\n$5.9\n$(1.8)\n$(8.8)\nQ1 2022 Q1 2023 |\n| --------------------------- |  | --------------------------------------- |  | --------------------------------------- |\n|                             |  |                                         |  |                                         |', 'dense': None, 'sparse': {334175660: 1.0, 485174231: 1.0, 806133968: 2.0, 822911587: 3.0, 839689206: 1.0, 856466825: 1.0, 873244444: 2.0, 923577301: 1.0, 1007465396: 1.0, 1024243015: 3.0, 1126777420: 3.0, 1143555039: 3.0, 2212577440: 1.0, 2262910297: 1.0, 2347784130: 1.0, 2414894606: 1.0, 2622263703: 6.0}}}, {'id': 137, 'distance': 0.019607843831181526, 'entity': {'id': 137, 'year': 2023, 'quarter': 'Q1', 'ticker': 'INTC', 'content': '|                           | $1.4\n$

In [8]:
res

data: ["{'id': 3, 'content': 'Apple Inc. (Exact name of Registrant as specified in its charter)'}", "{'id': 4, 'content': 'California 94-2404110 (State or other jurisdiction of incorporation or organization) (I.R.S. Employer Identification No.)'}", "{'id': 5, 'content': 'One Apple Park Way Cupertino, California 95014 (Address of principal executive offices) (Zip Code) (408) 996-1010 (Registrant’s telephone number, including area code)'}", "{'id': 6, 'content': 'Securities registered pursuant to Section 12(b) of the Act: Title of each class Trading symbol(s) Name of each exchange on which registered Common Stock, $0.00001 par value per share AAPL The Nasdaq Stock Market LLC 1.375% Notes due 2024 — The Nasdaq Stock Market LLC 0.000% Notes due 2025 — The Nasdaq Stock Market LLC 0.875% Notes due 2025 — The Nasdaq Stock Market LLC 1.625% Notes due 2026 — The Nasdaq Stock Market LLC 2.000% Notes due 2027 — The Nasdaq Stock Market LLC 1.375% Notes due 2029 — The Nasdaq Stock Market LLC 3.050%